# LNL on GTSRB — Nhóm X (plug-and-play)

Notebook gốc `Instructions.ipynb`, chỉ sửa **đường train cho LNL** (giữ nguyên phần tải dữ liệu, build model,
MoEx, FLOPs): clone+cd về repo nhóm (chứa `LNL.py` cải tiến), và áp dụng recipe đã đạt **99.56%**:
**AdamW (lr 5e-4, wd 0.05) + cosine LR + warmup + label smoothing 0.1 + AMP + EMA**, batch 64, 20 epoch,
và **TTA đa tỉ lệ** lúc test. Ảnh vào vẫn raw `[0,1]` (chuẩn hoá nằm trong `LNL.py`).

**Chạy:** Runtime → GPU (T4) → **Run all** (~1.5–2h). Số báo cáo là **`Standard accuracy`** ở cell Test.


In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/bnqtoan/LNL-GTSRB-NhomX.git

In [ ]:
cd /content/LNL-GTSRB-NhomX

In [ ]:
pip install torchattacks

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision.utils
from torchvision import models
import torchvision.datasets as dsets
import torchvision.transforms as transforms

import torchattacks
from torchattacks import PGD, FGSM
from torchsummary import summary

In [ ]:
print("PyTorch", torch.__version__)
print("Torchvision", torchvision.__version__)
print("Torchattacks", torchattacks.__version__)
print("Numpy", np.__version__)

## GTSRB

In [ ]:
!mkdir data

!curl --url https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip -o data/GTSRB_Final_Training_Images.zip
!curl --url https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip -o data/GTSRB_Final_Test_Images.zip
!curl --url https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_GT.zip -o data/GTSRB_Final_Test_GT.zip

In [ ]:
!unzip data/GTSRB_Final_Training_Images.zip -d data/ > /dev/null 2>&1
!unzip data/GTSRB_Final_Test_Images.zip -d data/ > /dev/null 2>&1
!unzip data/GTSRB_Final_Test_GT.zip -d data/

In [ ]:
import shutil

In [ ]:
data_dir = './data/GTSRB'
images_dir = os.path.join(data_dir, 'Final_Test/Images')

test_dir = os.path.join(data_dir, 'test')
os.makedirs(test_dir, exist_ok=True)



with open('./data/GT-final_test.csv') as f:
  image_names = f.readlines()

for text in image_names[1:]:
  classes = int(text.split(';')[-1])
  image_name = text.split(';')[0]
  

  test_class_dir = os.path.join(test_dir, f"{classes:04d}")
  os.makedirs(test_class_dir, exist_ok=True)
  image_path = os.path.join(images_dir, image_name)

  shutil.copy(image_path, test_class_dir)

In [ ]:
#Affine = transforms.RandomApply([transforms.RandomAffine(degrees=(0, 30),shear=(0.1, 0.2))], p=0.7)
#GaussianBlur = transforms.RandomApply([transforms.GaussianBlur((3,7) , sigma=(5))], p=0.7)
#Perspective = transforms.RandomPerspective(0.5, p = 0.7) 
#HSV1 = transforms.RandomApply([transforms.ColorJitter(brightness=0.3, contrast=.2)], p=0.7)
#HSV2 = transforms.RandomApply([transforms.ColorJitter(saturation= .2, hue=0.2)], p=0.7)
#miror = transforms.RandomHorizontalFlip(p=0.7)
#Randomcrop = transforms.RandomApply([transforms.RandomResizedCrop(100)], p=0.7)

#transform_list = [GaussianBlur, Affine, Perspective, HSV1, HSV2, Randomcrop]
                   

In [ ]:
batch_size = 64   # was 15: AdamW + larger batch is the proven recipe (raw [0,1]; LNL.py normalises in-model)

trainset = torchvision.datasets.ImageFolder(root='./data/GTSRB/Final_Training/Images',
                                                transform=transforms.Compose([
                                                          transforms.Resize((224,224)),
                                                          transforms.ToTensor(),
                                                          ]),
                                               )

testset = torchvision.datasets.ImageFolder(root='./data/GTSRB/test',
                                                transform=transforms.Compose([
                                                          transforms.Resize((224,224)),
                                                          transforms.ToTensor(),
                                                          ]),
                                               )

train_loader = torch.utils.data.DataLoader(dataset=trainset, batch_size=batch_size,
                                         shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

test_loader = torch.utils.data.DataLoader(dataset=testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
batch = next(iter(train_loader))
train_data = batch[0]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def normalize_image(image):
    image_min = image.min()
    image_max = image.max()
    image.clamp_(min = image_min, max = image_max)
    image.add_(-image_min).div_(image_max - image_min + 1e-5)
    return image

def plot_images(images, labels, classes, normalize=True):

    n_images = len(images)

    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))

    fig = plt.figure(figsize=(20, 20))

    for i in range(rows*cols):

        ax = fig.add_subplot(rows, cols, i+1)
        
        image = images[i]

        if normalize:
            image = normalize_image(image)

        ax.imshow(image.permute(1, 2, 0).cpu().numpy())
        ax.set_title(classes[labels[i]])
        ax.axis('off')

In [ ]:
classes = trainset.classes

plot_images(batch[0], batch[1], classes)

## model

In [ ]:
pip install timm

In [ ]:
pip install einops

In [ ]:
from LNL import LNL_Ti as small

In [ ]:
model = small(pretrained=False)

In [ ]:
model.head

In [ ]:
model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)

In [ ]:
model = model.cuda()

## Train Locality-iN-Locality

In [ ]:
num_epochs = 20

In [ ]:
import math, copy
# ---- proven recipe: AdamW + cosine LR (warmup) + label smoothing + AMP + EMA ----
WARMUP = 2
BASE_LR = 5e-4
loss = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=0.05)
scaler = torch.amp.GradScaler('cuda')

def lr_at(epoch):
    if epoch < WARMUP:
        return BASE_LR * (epoch + 1) / WARMUP
    p = (epoch - WARMUP) / max(1, (num_epochs - WARMUP))
    return 0.5 * BASE_LR * (1 + math.cos(math.pi * p))

# exponential moving average of the weights (evaluated alongside the raw model)
ema = copy.deepcopy(model).eval()
for p in ema.parameters():
    p.requires_grad_(False)
EMA_DECAY = 0.999
@torch.no_grad()
def ema_update():
    for e, m in zip(ema.parameters(), model.parameters()):
        e.mul_(EMA_DECAY).add_(m.detach(), alpha=1 - EMA_DECAY)
    for e, m in zip(ema.buffers(), model.buffers()):
        e.copy_(m)

In [ ]:
for epoch in range(num_epochs):
    lr = lr_at(epoch)
    for g in optimizer.param_groups:
        g['lr'] = lr

    total_batch = len(trainset) // batch_size
    model.train()

    for i, (batch_images, batch_labels) in enumerate(train_loader):
        X = batch_images.cuda(non_blocking=True)
        Y = batch_labels.cuda(non_blocking=True)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            pre = model(X)
            cost = loss(pre, Y)
        scaler.scale(cost).backward()
        scaler.step(optimizer)
        scaler.update()
        ema_update()

        if (i+1) % 200 == 0:
            print('Epoch [%d/%d], lter [%d/%d], lr %.2e, Loss: %.6f'
                 %(epoch+1, num_epochs, i+1, total_batch, lr, cost.item()))

## Test

In [ ]:
import torchvision.transforms.functional as TF

@torch.no_grad()
def evaluate(net):
    net.eval(); correct = 0; total = 0
    for images, labels in test_loader:
        images = images.cuda()
        with torch.amp.autocast('cuda'):
            outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels.cuda()).sum().item()
    return 100 * float(correct) / total

# pick the better of raw vs EMA
acc_raw = evaluate(model)
acc_ema = evaluate(ema)
best = model if acc_raw >= acc_ema else ema
print('raw: %.2f %% | EMA: %.2f %% -> using %s' % (acc_raw, acc_ema, 'raw' if best is model else 'EMA'))

# multi-scale test-time augmentation for the final Top-1
IMG = 224
@torch.no_grad()
def evaluate_tta(net):
    net.eval(); correct = 0; total = 0
    for images, labels in test_loader:
        images = images.cuda(); logits = 0
        for s in [1.0, 0.9, 1.1]:
            if s == 1.0:
                im = images
            else:
                n = int(IMG * s)
                im = TF.resize(images, [n, n])
                im = TF.center_crop(im, [IMG, IMG]) if s > 1 else TF.resize(im, [IMG, IMG])
            with torch.amp.autocast('cuda'):
                logits = logits + torch.softmax(net(im).float(), 1)
        pred = logits.argmax(1)
        total += labels.size(0); correct += (pred == labels.cuda()).sum().item()
    return 100 * float(correct) / total

print('Standard accuracy: %.2f %%' % evaluate_tta(best))

## FGSM attack

In [ ]:
model.eval()

correct = 0
total = 0

atk = FGSM(model, eps=0.01)

for images, labels in test_loader:
    
    images = atk(images, labels).cuda()
    outputs = model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()
    
print('Robust accuracy: %.2f %%' % (100 * float(correct) / total))

## PGD attack

In [ ]:
model.eval()

correct = 0
total = 0

atk = PGD(model, eps=0.01, alpha=2/255, steps=5, random_start=False)

for images, labels in test_loader:
    
    images = atk(images, labels).cuda()
    outputs = model(images)
    
    _, predicted = torch.max(outputs.data, 1)
    
    total += labels.size(0)
    correct += (predicted == labels.cuda()).sum()
    
print('Robust accuracy: %.2f %%' % (100 * float(correct) / total))

## train LNL-MoEx

In [ ]:
from LNL_MoEx import LNL_MoEx_Ti as small
model = small(pretrained=False)
model.head = torch.nn.Linear(in_features=192, out_features=43, bias=True)

In [ ]:
model = model.cuda()

In [ ]:
import time
# time.clock_gettime()

In [ ]:
num_epochs = 5
moex_lam = .9
moex_prob = .7

In [ ]:
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.007, momentum=0.9)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

In [ ]:
for epoch in range(num_epochs):

    total_batch = len(trainset) // batch_size
    
    for i, (input, target) in enumerate(train_loader):
        input = input.cuda()
        target = target.cuda()

        prob = torch.rand(1).item()
        if prob < moex_prob:
            swap_index = torch.randperm(input.size(0), device=input.device)
            with torch.no_grad():
                target_a = target
                target_b = target[swap_index]
            output = model(input, swap_index=swap_index, moex_norm='pono', moex_epsilon=1e-5,
                           moex_layer='stem', moex_positive_only=False)
            lam = moex_lam
            cost = loss(output, target_a) * lam + loss(output, target_b) * (1. - lam)
        else:
            # compute output
            output = model(input)
            # if args.prof >= 0: torch.cuda.nvtx.range_pop()
            cost = loss(output, target)

        # compute gradient and do SGD step

        optimizer.zero_grad()
        cost.backward()
        optimizer.step()

        if (i+1) % 200 == 0:
            print('Epoch [%d/%d], lter [%d/%d], Loss: %.6f'
                 %(epoch+1, num_epochs, i+1, total_batch, cost.item()))

## Number of Parameters

In [ ]:
pip install ptflops

In [ ]:
pip install --upgrade git+https://github.com/sovrasov/flops-counter.pytorch.git

In [ ]:
import torch
from ptflops import get_model_complexity_info

with torch.cuda.device(0):
  net = model
  macs, params = get_model_complexity_info(net, (3, 224, 224), as_strings=True,
                                           print_per_layer_stat=True, verbose=True)
  print('{:<30}  {:<8}'.format('Computational complexity: ', macs))
  print('{:<30}  {:<8}'.format('Number of parameters: ', params))
